# 04 — Kalibrasi Threshold (FAR ≤ 1%)**Tujuan:** menentukan **ambang similarity** yang dipakai saat inferensi: similarity di atas ambang → dikenali sebagai orang terdaftar; di bawah ambang → `"Tidak dikenal"`.**Mengapa wajib dihitung, bukan ditebak?** PRD §8.4 no. 5: keyakinan model yang tidak terkalibrasi menghasilkan keputusan salah dengan percaya diri tinggi. Ambang harus berasal dari **pengukuran FAR (False Accept Rate)** dan **TAR/FRR** pada data validasi.**Prasyarat** — notebook 03 sudah dijalankan:- `Kapita\gallery.npz` (centroid 69 × 512)- `Kapita\gallery_meta.json`- `Kapita_aug\embeddings_cache.npz` (embedding seluruh 17.680 file beserta label split)**Urutan eksekusi:** SEL 1 → SEL 9 berurutan, dari kernel kosong.**Data yang dipakai:** `split='val'` saja (70 grup → 1.820 file). **Test tidak boleh dipakai** untuk choosing threshold, kalau tidak evaluasi di notebook 05 jadi tidak jujur.**Dua definisi FAR yang dihitung (keduanya ditampilkan):**| Definisi | Arti | Sifat ||---|---|---|| `FAR_maks` | val vs **centroid orang lain dengan similarity tertinggi** | sangat konservatif (worst case per gambar) || `FAR_rata` | val vs **seluruh** 68 centroid lain (≈124 ribu percobaan) | mendekati kondisi operasional |**Kriteria pilihan ambang (PRD §9.5):** `FAR_rata ≤ 1%` dengan TAR maksimal; `FAR_maks` dilaporkan sebagai informationextra keamanan.**Output:** `Kapita\threshold.txt` (nilai + metadata) — dibaca notebook 05 & 06 (FR-6: bisa diubah tanpa mengubah kode inti).

In [ ]:
# SEL 1 — Impor paket + konstanta
import os
import sys
import json
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

DATA_ROOT = r"./data_riset/raw"
AUG_ROOT = r"./data_riset/aug"
GALLERY_NPZ = os.path.join(DATA_ROOT, "gallery.npz")
GALLERY_META = os.path.join(DATA_ROOT, "gallery_meta.json")
CACHE_NPZ = os.path.join(AUG_ROOT, "embeddings_cache.npz")
THRESHOLD_FILE = os.path.join(DATA_ROOT, "threshold.txt")
LAPORAN_FILE = os.path.join(DATA_ROOT, "kalibrasi_laporan.json")

# target keamanan (PRD §9.5 & NFR-2)
FAR_TARGET = 0.01          # FAR rata-rata maksimal 1%
GRID = np.arange(0.0, 1.0001, 0.005)
SEED = 42
np.random.seed(SEED)

print("Python         :", sys.version.split()[0])
print("gallery.npz    :", os.path.isfile(GALLERY_NPZ))
print("cache          :", os.path.isfile(CACHE_NPZ))
print("threshold.txt  :", os.path.isfile(THRESHOLD_FILE), "(boleh sudah ada -> ditimpa)")

In [ ]:
# SEL 2 — Muat gallery + cache embedding
g = np.load(GALLERY_NPZ, allow_pickle=False)
centroid = g["embeddings"].astype(np.float32)
person_ids = g["person"].astype(str)
aktif = g["aktif"].astype(bool)
model_name = str(g["model"][0])

with open(GALLERY_META, encoding="utf-8") as fh:
    meta = json.load(fh)

c = np.load(CACHE_NPZ, allow_pickle=False)
E = c["embedding"].astype(np.float32)
P = c["person"].astype(str)
SPLIT = c["split"].astype(str)
VARIAN = c["varian"].astype(str)
GRUP = c["grup"].astype(str)
PATH = c["path"].astype(str)

En = E / np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-9)

print("centroid       :", centroid.shape, "| model:", model_name)
print("orang aktif    :", int(aktif.sum()), "/", len(person_ids))
print("cache embedding:", En.shape)
print("split di cache :", {s: int((SPLIT == s).sum()) for s in sorted(set(SPLIT.tolist()))})
print("metadata       :", meta["jumlah_orang"], "orang, dibuat", meta["dibuat"])

assert En.shape[1] == centroid.shape[1], "dimensi embedding cache != dimensi gallery"
assert np.allclose(np.linalg.norm(centroid, axis=1), 1.0, atol=1e-4), "centroid belum ternormalisasi"

In [ ]:
# SEL 3 — Siapkan data validasi (split='val') + skor genuine/impostor
idx_person = {p: i for i, p in enumerate(person_ids)}
idx_valid = np.where(SPLIT == "val")[0]
print("file validasi  :", len(idx_valid))
print("orang di validasi:", len(set(P[idx_valid].tolist())))
print("grup di validasi :", len(set(zip(P[idx_valid].tolist(), GRUP[idx_valid].tolist()))))

# similarity ke SELURUH centroid, hanya untuk centroid yang aktif
cent_aktif = centroid[aktif]
person_aktif = person_ids[aktif]
col_aktif = np.where(aktif)[0]
remap = -np.ones(len(person_ids), dtype=np.int64)
for i, col in enumerate(col_aktif):
    remap[col] = i

rows = []
for k in idx_valid:
    e = En[k]
    sim = cent_aktif @ e                       # (n_aktif,)
    col_asli = idx_person[P[k]]
    pos = remap[col_asli]
    if pos < 0:
        continue                               # orang ini berstatus nonaktif
    genuine = float(sim[pos])
    others = sim.copy()
    others[pos] = -1.0
    impostor_maks = float(others.max())
    impostor_rata = float(others.mean())
    impostor_med = float(np.median(others))
    top2 = np.sort(sim)[::-1][:2]
    rows.append({"person": P[k], "grup": GRUP[k], "varian": VARIAN[k], "path": PATH[k],
                 "genuine": genuine, "impostor_maks": impostor_maks,
                 "impostor_rata": impostor_rata, "impostor_med": impostor_med,
                 "top1": float(top2[0]), "top2": float(top2[1])})

df_v = pd.DataFrame(rows)
print("\nbaris tabel validasi:", len(df_v))
print("\nDistribusi similarity pada VALIDASI:")
print(df_v[["genuine", "impostor_maks", "impostor_rata"]].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(4).to_string())
print("\nTop-1 tanpa ambang (identitas tertinggi): {:.2%}".format(
    float((df_v["top1"] == df_v["genuine"]).mean())))
print("Top-1 benar pada foto BERSIH (varian 'orig'): {:.2%}".format(
    float((df_v[df_v["varian"] == "orig"]["top1"] ==
           df_v[df_v["varian"] == "orig"]["genuine"]).mean())))

In [ ]:
# SEL 4 — Sweep threshold: hitung FAR & TAR (PRD §9.5)
genuine = df_v["genuine"].to_numpy()
imp_maks = df_v["impostor_maks"].to_numpy()

# FAR_rata: seluruh pasangan val x centroid lain (bukan hanya yang tertinggi)
# diambil secara acak 40.000 percobaan agar cepat tapi representatif
rng = np.random.default_rng(SEED)
n_imp = 40000
baris = rng.integers(0, len(df_v), size=n_imp)
kol = rng.integers(0, len(cent_aktif), size=n_imp)
tabel = df_v[["person"]].to_numpy()
sama = np.array([tabel[b][0] == person_aktif[j] for b, j in zip(baris, kol)])
sisa = ~sama
baris, kol = baris[sisa], kol[sisa]
imp_pairs = np.einsum("ij,ij->i", cent_aktif[kol], En[idx_valid[baris]])
print("percobaan impostor FAR_rata :", len(imp_pairs))

baris_sweep = []
for th in GRID:
    tar = float((genuine >= th).mean())
    frr = 1.0 - tar
    far_maks = float((imp_maks >= th).mean())
    far_rata = float((imp_pairs >= th).mean())
    baris_sweep.append({"threshold": float(th), "TAR": tar, "FRR": frr,
                        "FAR_maks": far_maks, "FAR_rata": far_rata})
df_sweep = pd.DataFrame(baris_sweep)

# EER (titik FAR = FRR) memakai FAR_maks sebagai kriteria konservatif
eer_idx = int(np.argmin(np.abs(df_sweep["FAR_maks"] - df_sweep["FRR"]).to_numpy()))
eer = df_sweep.iloc[eer_idx]

layanan = df_sweep[df_sweep["FAR_rata"] <= FAR_TARGET]
if len(layanan) > 0:
    terpilih = layanan.iloc[int(np.argmax(layanan["TAR"].to_numpy()))]
    aturan = "FAR_rata <= {:.1%} dengan TAR maksimal".format(FAR_TARGET)
else:
    terpilih = df_sweep.iloc[int(np.argmax(df_sweep["TAR"].to_numpy()))]
    aturan = "TIDAK ada ambang dengan FAR_rata <= {:.1%}; dipakai TAR maksimal".format(FAR_TARGET)

print("EER (FAR_maks = FRR) : threshold {:.3f} | FAR {:.2%} | FRR {:.2%}".format(
    eer["threshold"], eer["FAR_maks"], eer["FRR"]))
print("\nAMBANG TERPILIH")
print("  aturan     :", aturan)
print("  threshold  : {:.3f}".format(terpilih["threshold"]))
print("  TAR        : {:.2%}".format(terpilih["TAR"]))
print("  FRR        : {:.2%}".format(terpilih["FRR"]))
print("  FAR_rata   : {:.3%}".format(terpilih["FAR_rata"]))
print("  FAR_maks   : {:.3%}".format(terpilih["FAR_maks"]))

print("\nbeberapa ambang lain:")
for th in [0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8]:
    row = df_sweep.iloc[int(np.argmin(np.abs(df_sweep["threshold"] - th)))]
    print("  th {:.2f} | TAR {:6.2%} | FRR {:6.2%} | FAR_rata {:7.3%} | FAR_maks {:6.2%}".format(
        row["threshold"], row["TAR"], row["FRR"], row["FAR_rata"], row["FAR_maks"]))

In [ ]:
# SEL 5 — Visualisasi kurva FAR/TAR + histogram
th_terpilih = float(terpilih["threshold"])

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(df_sweep["threshold"], df_sweep["TAR"], label="TAR (penerimaan benar)", lw=2)
ax.plot(df_sweep["threshold"], 1 - df_sweep["TAR"], label="FRR (penolakan salah)", lw=2)
ax.plot(df_sweep["threshold"], df_sweep["FAR_rata"], label="FAR rata-rata", lw=2)
ax.plot(df_sweep["threshold"], df_sweep["FAR_maks"], label="FAR max (konservatif)", lw=2)
ax.axvline(th_terpilih, color="red", ls="--", label="ambang dipilih = {:.3f}".format(th_terpilih))
ax.set_xlabel("threshold similarity")
ax.set_ylabel("tingkat")
ax.set_title("Kalibrasi threshold pada split VAL (70 grup / {} file)".format(len(df_v)))
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
bins = np.arange(0, 1.01, 0.02)
ax.hist(df_v["impostor_maks"], bins=bins, alpha=0.65, label="impostor (maks thd centroid lain)", density=True)
ax.hist(df_v["genuine"], bins=bins, alpha=0.65, label="genuine (centroid sendiri)", density=True)
ax.axvline(th_terpilih, color="red", ls="--", label="ambang = {:.3f}".format(th_terpilih))
ax.set_xlabel("cosine similarity")
ax.set_title("Sebaran similarity di split VAL")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# versi logaritmik untuk melihat ekor impostor
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(df_v["impostor_maks"], bins=np.arange(0, 1.01, 0.02), alpha=0.65,
        label="impostor maks", density=True)
ax.hist(df_v["genuine"], bins=np.arange(0, 1.01, 0.02), alpha=0.65,
        label="genuine", density=True)
ax.set_yscale("log")
ax.set_xlabel("cosine similarity")
ax.set_title("Sebaran similarity (sumbu Y log)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# SEL 6 — Simpan threshold.txt + laporan (FR-6: bisa diubah tanpa ubah kode)
isi = {
    "threshold": th_terpilih,
    "aturan": aturan,
    "target_far_rata": FAR_TARGET,
    "TAR": float(terpilih["TAR"]),
    "FRR": float(terpilih["FRR"]),
    "FAR_rata": float(terpilih["FAR_rata"]),
    "FAR_maks": float(terpilih["FAR_maks"]),
    "eer_threshold": float(eer["threshold"]),
    "eer_far": float(eer["FAR_maks"]),
    "eer_frr": float(eer["FRR"]),
    "split": "val",
    "n_validasi": int(len(df_v)),
    "jumlah_orang": int(len(person_ids)),
    "model": model_name,
    "dibuat": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}
dengan_komentar = "{:.6f}\n".format(th_terpilih)
dengan_komentar += "# ---------------------------------------------------------------\n"
dengan_komentar += "# Ambang similarity untuk sistem absensi (PRD §9.5)\n"
dengan_komentar += "# BARIS PERTAMA = nilai ambang yang dipakai notebook 05 & 06.\n"
dengan_komentar += "# Baris 'key=value' di bawah hanya metadata untuk pelaporan.\n"
dengan_komentar += "# FAR = false accept rate, TAR = true accept rate, FRR = false reject rate\n"
dengan_komentar += "# ---------------------------------------------------------------\n"
for k, v in isi.items():
    dengan_komentar += "{}={}\n".format(k, v)

with open(THRESHOLD_FILE, "w", encoding="utf-8") as fh:
    fh.write(dengan_komentar)
with open(LAPORAN_FILE, "w", encoding="utf-8") as fh:
    json.dump({"ringkasan": isi, "sweep": df_sweep.to_dict("records")}, fh,
              ensure_ascii=False, indent=2)
df_sweep.to_csv(os.path.join(DATA_ROOT, "kalibrasi_sweep.csv"), index=False, encoding="utf-8-sig")

print("tersimpan:")
print("  threshold.txt         :", THRESHOLD_FILE)
print("  kalibrasi_laporan.json:", LAPORAN_FILE)
print("  kalibrasi_sweep.csv   :", os.path.join(DATA_ROOT, "kalibrasi_sweep.csv"))
print("\nisi threshold.txt:")
print(dengan_komentar)


def baca_threshold(path=THRESHOLD_FILE):
    """Fungsi yang dipakai notebook 05 & 06 (FR-6)."""
    nilai = None
    meta_th = {}
    with open(path, encoding="utf-8") as fh:
        for baris in fh:
            baris = baris.strip()
            if not baris or baris.startswith("#"):
                continue
            if "=" in baris:
                k, v = baris.split("=", 1)
                meta_th[k.strip()] = v.strip()
            else:
                nilai = float(baris)
    return nilai, meta_th


nilai, meta_th = baca_threshold()
print("dibaca kembali ->", nilai, "| TAR", meta_th.get("TAR"), "| FAR_rata", meta_th.get("FAR_rata"))
assert nilai is not None and 0.0 <= nilai <= 1.0, "threshold.txt tidak valid"
assert abs(nilai - th_terpilih) < 1e-6, "nilai tersimpan tidak cocok dengan ambang terpilih"

In [ ]:
# SEL 7 — Analisis per orang pada ambang terpilih (siapa yang bermasalah)
df_v["terima"] = df_v["genuine"] >= th_terpilih
df_v["salah_identitas"] = df_v["terima"] & (df_v["top1"] > df_v["genuine"])

per_orang = df_v.groupby("person").agg(
    n_file=("genuine", "size"),
    tar=("terima", "mean"),
    mean_genuine=("genuine", "mean"),
    p05_genuine=("genuine", lambda s: np.percentile(s, 5)),
    n_gagal=("terima", lambda s: int((~s).sum())),
    n_salah_id=("salah_identitas", "sum"),
).reset_index()
per_orang["nama"] = per_orang["person"].map({o["person"]: o["nama"] for o in meta["orang"]})
per_orang["nim"] = per_orang["person"].map({o["person"]: o["nim"] for o in meta["orang"]})
per_orang["kelas"] = per_orang["person"].map({o["person"]: o["kelas"] for o in meta["orang"]})

print("TAR keseluruhan pada ambang {:.3f}: {:.2%}".format(th_terpilih, df_v["terima"].mean()))
print("\n10 orang dengan TAR terendah (kandidat perlu foto tambahan / review):")
print(per_orang.nsmallest(10, "tar")[
    ["nama", "nim", "kelas", "n_file", "tar", "mean_genuine", "p05_genuine", "n_gagal"]
].to_string(index=False))

print("\nkasus salah identitas (terima tapi similarity tertinggi milik orang lain):",
      int(df_v["salah_identitas"].sum()))
if int(df_v["salah_identitas"].sum()) > 0:
    salah = df_v[df_v["salah_identitas"]].nlargest(10, "top1")
    print(salah[["person", "grup", "varian", "genuine", "top1", "impostor_maks"]].to_string(index=False))

#FAR per kelas? -> rekap jumlah penolakan per kelas
rekap_kelas = per_orang.groupby("kelas")["tar"].agg(["mean", "size"]).round(3)
print("\nTAR rata-rata per kelas:")
print(rekap_kelas.to_string())

per_orang.to_csv(os.path.join(DATA_ROOT, "kalibrasi_per_orang.csv"), index=False, encoding="utf-8-sig")
print("\ndisimpan:", os.path.join(DATA_ROOT, "kalibrasi_per_orang.csv"))

In [ ]:
# SEL 8 — Verifikasi ambang terhadap NFR-2 (FAR < 1%)
print("=" * 60)
print("VERIFIKASI NFR-2: FAR < 1% pada ambang operasional")
print("=" * 60)
print("threshold dipakai     :", th_terpilih)
print("FAR rata-rata (val)   : {:.3%}  {}".format(
    float(terpilih["FAR_rata"]), "OK" if terpilih["FAR_rata"] < 0.01 else "LEBIH"))
print("FAR maks (konservatif): {:.3%}".format(float(terpilih["FAR_maks"])))
print("TAR                   : {:.2%}".format(float(terpilih["TAR"])))
print("FRR                   : {:.2%}".format(float(terpilih["FRR"])))

# pemeriksaan tambahan: berapa file validasi yang akan ditolak
ditolak = int((~df_v["terima"]).sum())
ditolak_bersih = int((~df_v[df_v["varian"] == "orig"]["terima"]).sum())
n_bersih = int((df_v["varian"] == "orig").sum())
print("\nfile validasi ditolak  : {} / {} ({:.2%})".format(ditolak, len(df_v), ditolak / len(df_v)))
print("foto bersih ditolak  : {} / {} ({:.2%})".format(
    ditolak_bersih, n_bersih, ditolak_bersih / max(n_bersih, 1)))

assert float(terpilih["FAR_rata"]) <= 0.01, "FAR rata-rata melewati target 1%"
assert 0.0 < th_terpilih < 1.0, "threshold di luar rentang"
assert os.path.isfile(THRESHOLD_FILE), "threshold.txt tidak tersimpan"
print("\nOK: ambang lolos NFR-2 dan siap dipakai notebook 05 & 06.")
print("\nCATATAN untuk A3 (20 wajah asing): data wajah asing belum ada di laptop.")
print("Sistem harus diuji dengan 20 wajah non-siswa secara langsung (lihat notebook 05 SEL uji/webcam).")

# Catatanhandoff ke notebook 05 & 061. **Ambang ada di file, bukan di kode** — `threshold.txt` baris pertama berisi nilai numerik. Notebook 05 & 06 membacanya lewat fungsi `baca_threshold()`. Mengubah angka secara manual = mengubah perilaku sistem tanpa menyentuh kode (FR-6).2. **Metrik yang dipakai** — `FAR_rata` (±124 ribu percobaan impostor) adalah angka operasional; `FAR_maks` (worst case per gambar) selalu lebih besar dan bersifat pessimis. Keduanya ditampilkan agar tidak ada angka yang disembunyikan.3. **Angka ini diukur pada `split='val'`.** Test belum pernah disentuh → notebook 05 masih bisa_called jujur (A1).4. **Top-1 tanpa ambang** di val ≈ 92% (foto bersih) — ini indikasi bahwa identifikasi (1 banding 69) sehat. Verifikasi (1 banding 1) selalu lebih ketat karena harus memenuhi FAR bersamaan.5. **Langkah berikutnya:** `05_evaluasi.ipynb` — evaluasi honest pada `split='test'`, laporan per kelas, pasangan yang tertukar, uji webcam (A2), dan uji 20 wajah asing (A3).